# Infomap quickstart

This notebook is the shortest path from a Python graph to an Infomap result you can inspect, summarize, visualize, and reuse.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from infomap import Infomap

## Run Infomap on a familiar graph

Start with Zachary's Karate Club graph. It is small enough to inspect and familiar enough to make the partition easy to judge.

In [ ]:
graph = nx.karate_club_graph()

im = Infomap(silent=True, num_trials=20, seed=123)
node_mapping = im.add_networkx_graph(graph)
im.run()
im

The `Infomap` object summarizes the result in place. Codelength and relative savings describe compression. Module counts describe the partition. The flow strip shows how much flow belongs to the largest top modules.

In [ ]:
assignments = {
    node_mapping[node.state_id]: node.module_id
    for node in im.nodes
    if node.state_id in node_mapping
}

result = im.to_dataframe(
    columns=["node_id", "module_id", "flow", "path"],
    index="node_id",
).rename(index=node_mapping)
result.head()

## Draw a readable partition

This helper is notebook-local. It is meant to be copied, changed, or deleted. It is not a public Infomap plotting API.

In [ ]:
def draw_network_partition(graph, modules, *, title=None, seed=7, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))

    pos = nx.spring_layout(graph, seed=seed)
    module_ids = sorted(set(modules.values()))
    palette = plt.get_cmap("tab20")
    colors = {
        module_id: palette(index % 20) for index, module_id in enumerate(module_ids)
    }
    node_colors = [colors[modules[node]] for node in graph.nodes]

    nx.draw_networkx_edges(graph, pos, ax=ax, alpha=0.25, width=1.0)
    nx.draw_networkx_nodes(
        graph,
        pos,
        ax=ax,
        node_color=node_colors,
        node_size=120,
        linewidths=0.8,
        edgecolors="white",
    )
    nx.draw_networkx_labels(graph, pos, ax=ax, font_size=8)
    ax.set_title(title or "Infomap partition")
    ax.axis("off")
    return ax


draw_network_partition(
    graph,
    assignments,
    title=f"Karate Club: {im.num_top_modules} modules, {im.codelength:.3f} bits",
);

## Move from toy graph to research data

For larger networks, the first view should usually be a result summary and a table, not a force-directed drawing.

In [ ]:
jazz = Infomap(silent=True, num_trials=20, seed=123)
jazz.read_file("data/jazz.net")
jazz.run()
jazz

In [ ]:
jazz_nodes = jazz.to_dataframe(columns=["node_id", "module_id", "flow"])

module_summary = (
    jazz_nodes.groupby("module_id", as_index=False)
    .agg(nodes=("node_id", "count"), flow=("flow", "sum"))
    .sort_values("flow", ascending=False)
    .reset_index(drop=True)
)
module_summary.head(10)

## Use the quick function when you only need assignments

The `Infomap` object gives you the richest result. For scripts that only need a node-to-module mapping, use the NetworkX helper.

In [ ]:
from infomap import find_communities

communities = find_communities(graph, seed=123, num_trials=20)
pd.Series(communities, name="module_id").head()

## Export results

Use dataframes for analysis, annotated graph objects for Python workflows, and GraphML or GEXF when you want to continue in visualization tools.

In [ ]:
annotated_graph = graph.copy()
nx.set_node_attributes(annotated_graph, assignments, "infomap_module")

result.to_csv("output/karate-infomap-nodes.csv")
nx.write_graphml(annotated_graph, "output/karate-infomap.graphml")

## How to cite

If you use Infomap in published work, mapequation.org recommends citing two things: the map equation paper (Rosvall and Bergstrom, 2008, <https://doi.org/10.1073/pnas.0706851105>) for the method, and the MapEquation software package (Edler, Holmgren, and Rosvall) for the implementation and version. See [How to cite](https://www.mapequation.org/publications/) for BibTeX.

## Where to go next

- {doc}`options guide <options-guide>` explains every Infomap option.
- {doc}`benchmark-performance <benchmark-performance>` helps you plan run time and memory.
- The compare notebooks put Infomap next to Louvain and Leiden in
  {doc}`NetworkX <compare-infomap-louvain-networkx>`,
  {doc}`igraph <compare-infomap-louvain-leiden-igraph>`, and
  {doc}`Scanpy <compare-infomap-scanpy-workflow>` workflows.